# mesh-n-bone end-to-end example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janelia-cellmap/mesh-n-bone/blob/master/examples/example.ipynb)

Runs the full pipeline in one notebook:
1. Install the package
2. Create a 256³ zarr segmentation volume (sphere + torus + ridged blob)
3. Generate meshes and the multires Draco output
4. Launch a local neuroglancer viewer

Works on a local machine or in Google Colab. Skeletonization is **not** included — it requires a separately compiled CGAL binary.

## 1. Install

In [ ]:
%pip install --quiet 'mesh-n-bone @ git+https://github.com/janelia-cellmap/mesh-n-bone.git'

## 2. Create the example zarr volume

256³ uint8 volume with three labels:
- Label 1: sphere (radius 50)
- Label 2: torus (R=40, r=20)
- Label 3: ridged blob with high-frequency surface detail

Written as zarr v3 with OME-NGFF multiscales metadata.

In [ ]:
import json, os, shutil
import numpy as np
import tensorstore as ts

work_dir = os.path.abspath('mesh-n-bone-example')
zarr_path = os.path.join(work_dir, 'data', 'example.zarr')
seg_path = os.path.join(zarr_path, 'seg')
array_path = os.path.join(seg_path, 's0')

def reset_directory(path):
    if not os.path.exists(path):
        return
    try:
        shutil.rmtree(path)
    except OSError:
        stale_path = f'{path}.stale-{os.getpid()}'
        os.rename(path, stale_path)
        shutil.rmtree(stale_path, ignore_errors=True)

if os.path.exists(zarr_path):
    reset_directory(zarr_path)

vol = np.zeros((256, 256, 256), dtype=np.uint8)

def object_box(center, radius):
    if np.isscalar(radius):
        radius = (radius, radius, radius)
    begin = [max(0, int(c - r)) for c, r in zip(center, radius)]
    end = [min(s, int(c + r + 1)) for c, r, s in zip(center, radius, vol.shape)]
    return tuple(slice(b, e) for b, e in zip(begin, end))

def box_coords(box):
    z, y, x = box
    return np.ogrid[z.start:z.stop, y.start:y.stop, x.start:x.stop]

sphere_box = object_box((128, 64, 64), 50)
zz, yy, xx = box_coords(sphere_box)
sphere = vol[sphere_box]
sphere[(zz - 128)**2 + (yy - 64)**2 + (xx - 64)**2 <= 50**2] = 1

torus_box = object_box((128, 192, 192), (60, 60, 20))
zz, yy, xx = box_coords(torus_box)
dist_to_ring = (np.sqrt((yy - 192)**2 + (zz - 128)**2) - 40)**2 + (xx - 192)**2
torus = vol[torus_box]
torus[dist_to_ring <= 20**2] = 2

blob_box = object_box((128, 192, 64), 48)
zz, yy, xx = box_coords(blob_box)
dz, dy, dx = zz - 128, yy - 192, xx - 64
r = np.sqrt(dx**2 + dy**2 + dz**2)
azimuth = np.arctan2(dy, dx)
polar = np.arctan2(dz, np.sqrt(dx**2 + dy**2))
ridged_surface = (
    34
    + 6 * np.sin(7 * azimuth + 2 * np.sin(3 * polar))
    + 4 * np.sin(9 * polar)
    + 2 * np.sin(0.27 * dx + 0.17 * dy + 0.21 * dz)
)
blob = vol[blob_box]
blob[r <= ridged_surface] = 3

multiscales = [{
    'version': '0.5',
    'axes': [{'name': a, 'type': 'space', 'unit': 'nanometer'} for a in 'zyx'],
    'datasets': [{'path': 's0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0]}]}],
}]
os.makedirs(seg_path, exist_ok=True)
with open(os.path.join(zarr_path, 'zarr.json'), 'w') as f:
    json.dump({'zarr_format': 3, 'node_type': 'group', 'attributes': {}}, f)
with open(os.path.join(seg_path, 'zarr.json'), 'w') as f:
    json.dump({'zarr_format': 3, 'node_type': 'group', 'attributes': {'multiscales': multiscales}}, f)

arr = ts.open({
    'driver': 'zarr3',
    'kvstore': {'driver': 'file', 'path': array_path},
    'metadata': {
        'shape': list(vol.shape),
        'data_type': 'uint8',
        'chunk_grid': {'name': 'regular', 'configuration': {'chunk_shape': [64, 64, 64]}},
    },
    'create': True, 'delete_existing': True,
}).result()
arr.write(vol).result()
print(f'Wrote {array_path}, labels {np.unique(vol[vol > 0]).tolist()}')

## 3. Run meshify

Marching cubes → simplification → multiresolution Draco output. Calls the `Meshify` class directly so we don't need a config file.

In [ ]:
from mesh_n_bone.meshify.meshify import Meshify

output_dir = os.path.join(work_dir, 'output')
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

meshify = Meshify(
    input_path=array_path,
    output_directory=output_dir,
    num_workers=1,
    do_simplification=True,
    target_reduction=0.5,
    n_smoothing_iter=2,
    check_mesh_validity=False,
    do_multires=True,
    num_lods=3,
    multires_strategy='decimate',
    decimation_factor=2,
    # Lower than the production default (25,000) so these small toy meshes
    # split into enough chunks for multires loading to be apparent in the viewer.
    target_faces_per_lod0_chunk=2000,
    delete_decimated_meshes=True,
    do_analysis=False,
)
meshify.get_meshes()
print('Meshes written to', output_dir)

## 4. View in neuroglancer

Starts a Python `neuroglancer.Viewer` and serves the example files from that same viewer server. Keeping the viewer and data on one origin avoids Colab proxy CORS failures.

On Colab, click the printed `https://<id>.colab...` viewer URL. On a local machine, click the printed `http://localhost:...` URL.

In [ ]:
try:
    import google.colab.output
except Exception:
    pass
import neuroglancer
from mesh_n_bone.serve import _build_source_urls, _mount_neuroglancer_static_files

neuroglancer.set_server_bind_address('0.0.0.0')
viewer = neuroglancer.Viewer()
base = _mount_neuroglancer_static_files(
    work_dir,
    viewer.get_viewer_url(),
    route_id=viewer.token,
)
sources = _build_source_urls(
    work_dir,
    base,
    zarr_path='data/example.zarr/seg',
    meshes_path='output/multires',
)
with viewer.txn() as s:
    s.layers['segmentation'] = neuroglancer.SegmentationLayer(
        source=sources,
        segments=[1, 2, 3],
    )
print('Open this URL:')
print(' ', viewer.get_viewer_url())